# 1. Basic Queries and Intermediate Queries

## Import Required Libraries

In [11]:
import random
import pandas as pd
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

## Database Creation

In [4]:
import sqlite3
conn = sqlite3.connect("ecommerce.db")
print("Successfully Database Connected")
cursor = conn.cursor()

Successfully Database Connected


## Helper Function

In [5]:
def run_query(query):
   return pd.read_sql_query(query,conn)

## Table Creation

In [6]:
customers = pd.read_csv("customers.csv")
products = pd.read_csv("cleaned_products.csv")
orders = pd.read_csv("cleaned_orders.csv")
order_items = pd.read_csv("order_itmes.csv")

In [7]:
customers.to_sql("customers",conn,if_exists="replace",index=False)

products.to_sql("products",conn,if_exists="replace",index=False)

orders.to_sql("orders",conn,if_exists="replace",index=False)

order_items.to_sql("order_items",conn,if_exists="replace",index=False)

print("All tables loaded successfully.")

All tables loaded successfully.


# 1. Basic Queries

### Q1. Total revenue per category (revenue = quantity × unit_price × (1 - discount_percent/100))

In [8]:
q1 = """select p.category,sum(o.quantity * o.unit_price * (1 - o.discount_percent/100)) as total_revenue
        from order_items as o
        inner join products as p
        on o.product_id = p.product_id
        group by p.category
        order by total_revenue desc;
        """
df_q1 = run_query(q1)
df_q1

,category,total_revenue
0,Home,1514830
1,Clothing,1276214
2,Electronics,1043079
3,Books,978660


### Q2. Top 10 customers by total order value

In [9]:
q2 = """select c.customer_id, c.customer_name,
        round(sum(oi.quantity * oi.unit_price *(1 - oi.discount_percent / 100.0)),2) AS total_order_value
        from customers c
        join orders o
        on c.customer_id = o.customer_id
        join order_items oi
        on o.order_id = oi.order_id
        group by c.customer_id,c.customer_name
        order by total_order_value desc
        limit 10;"""
df_q2 = run_query(q2)
df_q2

,customer_id,customer_name,total_order_value
0,253,Christopher Shelton,45437.65
1,390,Hayley Graves,42537.44
2,352,Autumn Chen,39431.85
3,193,Austin Collier,37579.22
4,227,Shannon Davis,37503.10
5,168,Adam Roberts,36093.88
6,60,Jeffrey Richardson,35966.02
7,174,Danny Mcmahon,35946.81
8,405,Martha Reyes,32744.99
9,219,Jennifer Carey,32450.86


### Q3. Month-wise order count for the last 12 months

In [10]:
q3 = """select strftime('%Y-%m', order_date) as month,count(order_id) as count_orders
        from orders
        group by month
        order by month desc
        limit 12;"""
df_q3 = run_query(q3)
df_q3

,month,count_orders
0,2026-07,4
1,2026-06,22
2,2026-05,19
3,2026-04,24
4,2026-03,17
5,2026-02,15
6,2026-01,22
7,2025-12,23
8,2025-11,13
9,2025-10,20


## 2. Intermediate Queries

### Q4. Find customers who placed orders but never had any item delivered

In [12]:
q4 = """select distinct c.customer_id,c.customer_name
from customers c
left join orders o
on c.customer_id = o.customer_id
AND o.status = 'DELIVERED'
WHERE o.order_id IS NULL;"""

df_q4 = run_query(q4)
df_q4

,customer_id,customer_name
0,1,Ronnie Sullivan
1,2,Matthew Pena
2,3,Andrew Cook
3,4,Kelli Perez
4,6,Joyce Liu
...,...,...
398,493,Jeffery Gonzalez
399,494,Sara Duncan
400,497,Brittney Ponce
401,498,Emily Werner


### Q5. Products that were ordered but had more returns than purchases

In [13]:
q5 = """select p.product_id, p.product_name,
sum(case when o.status = 'RETURNED' then 1 else 0 end)as returns,
sum(case when o.status = 'DELIVERED' then 1 else 0 end)as purchases
from products p
join order_items oi
on p.product_id = oi.product_id
join orders o
on oi.order_id = o.order_id
group by p.product_id,p.product_name
having returns > purchases;"""
df_q5 = run_query(q5)
df_q5

,product_id,product_name,returns,purchases
0,26,Handbag,1,0
1,28,Pan,1,0
2,42,T-Shirt,1,0
3,50,Jeans,1,0
4,61,Macbook,1,0
...,...,...,...,...
63,486,Sofa,1,0
64,490,Sony,1,0
65,495,Macbook,2,0
66,498,Lenovo,1,0


## Q6. Calculate the return rate (returned items / total items) per category

In [14]:
q6 = """select p.category,
sum(case
    when o.status='RETURNED'then oi.quantity else 0
    end) as returned_items,
sum(oi.quantity) as total_items,
round(sum(case when o.status='RETURNED' then oi.quantity else 0
end) *100.0/ sum(oi.quantity),2) as return_rate
from products p
join order_items oi
on p.product_id=oi.product_id
join orders o
on oi.order_id=o.order_id
group by p.category;"""

df_q6 = run_query(q6)
df_q6

,category,returned_items,total_items,return_rate
0,Books,51,295,17.29
1,Clothing,68,367,18.53
2,Electronics,58,310,18.71
3,Home,72,425,16.94
